In [1]:
import os
import sys
import lsdb
import dask
import dask.distributed
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset
from upath import UPath

In [2]:
s3_bucket = "nasa-irsa-euclid-q1"
euclid_prefix = "contributed/q1/merged_objects/hats"

euclid_hats_collection_uri = f"s3://{s3_bucket}/{euclid_prefix}"  # for lsdb
euclid_parquet_metadata_path = f"{s3_bucket}/{euclid_prefix}/euclid_q1_merged_objects-hats/dataset/_metadata"  # for pyarrow

max_magnitude = 24.5
min_flux = 10 ** ((max_magnitude - 23.9) / -2.5)

In [3]:
s3_bucket = "nasa-irsa-euclid-q1"
euclid_prefix = "contributed/q1/merged_objects/hats"

euclid_hats_collection_uri = f"s3://{s3_bucket}/{euclid_prefix}"  # for lsdb
euclid_parquet_metadata_path = f"{s3_bucket}/{euclid_prefix}/euclid_q1_merged_objects-hats/dataset/_metadata"  # for pyarrow

max_magnitude = 24.5
min_flux = 10 ** ((max_magnitude - 23.9) / -2.5)

In [4]:
OBJECT_ID = "object_id"
PHZ_Z = "phz_phz_median"
columns = [OBJECT_ID, PHZ_Z]

In [5]:
phz_filter = (
        (pc.field("mer_vis_det") == 1)  # No NIR-only objects.
        & (pc.field("mer_flux_detection_total") > min_flux)  # I < 24.5
        & (pc.divide(pc.field("mer_flux_detection_total"),
                     pc.field("mer_fluxerr_detection_total")) > 5)  # I band S/N > 5
        & ~pc.field("phz_phz_classification").isin([1, 3, 5, 7])  # Exclude objects classified as star.
        & (pc.field("mer_spurious_flag") == 0)  # MER quality
)

# Load.
dataset = pyarrow.dataset.parquet_dataset(euclid_parquet_metadata_path, partitioning="hive",
                                          filesystem=pyarrow.fs.S3FileSystem())

In [6]:
pa_df = dataset.to_table(columns=columns, filter=phz_filter).to_pandas()

In [13]:
pa_full_df = dataset.to_table(columns=columns).to_pandas()

In [7]:
query = (
    "mer_vis_det == 1"
    f" & mer_flux_detection_total > {min_flux}"
    " & mer_flux_detection_total / mer_fluxerr_detection_total > 5"
    " & phz_phz_classification not in [1,3,5,7]"
    " & mer_spurious_flag == 0"
)

# We don't want to load these columns, but we have to in order to use them in the filter.
extra_columns = ["mer_vis_det", "mer_flux_detection_total", "phz_phz_classification", "mer_spurious_flag",
                 "mer_fluxerr_detection_total"]

In [23]:
lsdb_catalog = lsdb.read_hats(euclid_hats_collection_uri, columns=columns + extra_columns)

In [29]:
from distributed import Client

client = Client(n_workers=os.cpu_count())

/Users/smcmu/.pyenv/versions/lsdb-benchmarking/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57936 instead
  warnings.warn(
2026-02-18 18:16:33,634 - distributed.dashboard.components.scheduler - ERROR - 'str' object has no attribute 'text'
Traceback (most recent call last):
  File "/Users/smcmu/.pyenv/versions/lsdb-benchmarking/lib/python3.12/site-packages/distributed/utils.py", line 829, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/smcmu/.pyenv/versions/lsdb-benchmarking/lib/python3.12/site-packages/distributed/dashboard/components/scheduler.py", line 442, in update
    self.root.title.text = title
    ^^^^^^^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'text'
2026-02-18 18:16:33,636 - distributed.dashboard.components.scheduler - ERROR - 'str' object has no attribute 'text'
Traceback (most recent cal

In [30]:
lsdb_df = lsdb_catalog.query(query).map_partitions(lambda df: df.iloc[:1]).compute()

/Users/smcmu/code/lsdb/src/lsdb/catalog/dataset/healpix_dataset.py:1023: RuntimeWarning: output of the function must be a DataFrame to generate an LSDB `Catalog`. `map_partitions` will return a dask object instead of a Catalog.
  warnings.warn(


In [28]:
client.shutdown()
client.close()